# 03 – Python Toolkit, Tests and Clean-Clone Check (S3)

**Course section:** Python Programlama (+ Git/GitHub bonus).

**Goal:** turn repeated notebook code into small, tested Python modules, and prove that the repository runs from a fresh copy.

| Course topic | Where it is used in this project |
|---|---|
| Data structures (list, tuple, dict, set) | rule tuples, amenity lists, `BOOL_MAP` dict, sets of neighbourhoods |
| Functions | `src/data_utils.py`: `load_city_listings`, `to_bool`, `parse_amenities`, `missing_report` |
| Control flow (`if`, `for`) | column detection, test runner, quality checks |
| Classes | `src/quality.py`: `DataQualityChecker`, `RangeRule` (dataclass), custom `DataQualityError` |
| Functional programming | `map`, `filter`, `lambda`, `Counter` + `chain` on amenity lists |
| Modules | `src/` and `configs/` imported by every notebook |
| Error handling | clear errors for missing files / unknown cities / unknown columns; `try/except` in the test runner |

In [1]:
# --- Imports and settings ---
import sys
from collections import Counter
from itertools import chain
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from configs.config import CITY, RANDOM_STATE
from src.data_utils import find_tf_columns, load_city_listings, missing_report, parse_amenities, to_bool
from src.quality import NYC_RULES, DataQualityChecker, DataQualityError, RangeRule

TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)

df = load_city_listings(CITY)
print(CITY, df.shape)

New York (37012, 33)


## 1. Functions and data structures
`find_tf_columns` uses a **set** comparison to detect the t/f columns; `to_bool` converts them with a **dict** lookup.

In [2]:
tf_cols = find_tf_columns(df)
print("t/f columns:", tf_cols)

df_b = to_bool(df, tf_cols)
print(df_b[tf_cols].dtypes.to_string())
df_b[tf_cols].mean().round(3).rename("share True").to_frame()   # booleans average to a share

t/f columns: ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
host_is_superhost         boolean
host_has_profile_pic      boolean
host_identity_verified    boolean
instant_bookable          boolean


,share True
host_is_superhost,0.188
host_has_profile_pic,0.997
host_identity_verified,0.8
instant_bookable,0.298


## 2. Functional programming on the amenity lists
`parse_amenities` turns the text into a **list**; `map` applies it to every row, `lambda` counts, `chain` + `Counter` find the most common amenities.

In [3]:
amenity_lists = df["amenities"].map(parse_amenities)
n_amenities = amenity_lists.map(lambda lst: len(lst))
print("Listings with no parsable amenities:", int((n_amenities == 0).sum()))
print(n_amenities.describe().round(1).to_string())

top = Counter(chain.from_iterable(amenity_lists)).most_common(10)
pd.DataFrame(top, columns=["amenity", "n_listings"]).assign(
    share=lambda t: (t["n_listings"] / len(df)).round(3))

Listings with no parsable amenities: 1
count    37012.0
mean        20.5
std          9.1
min          0.0
25%         13.0
50%         19.0
75%         27.0
max         78.0


,amenity,n_listings,share
0,Wifi,35904,0.970
1,Long term stays allowed,34838,0.941
2,Heating,34787,0.940
3,Kitchen,33619,0.908
4,Essentials,33605,0.908
5,Smoke alarm,32430,0.876
6,Air conditioning,31544,0.852
7,Hangers,28048,0.758
8,Carbon monoxide alarm,26892,0.727
9,TV,25835,0.698


In [4]:
# filter + lambda: amenities that look like a workspace (relevant for long stays during COVID)
all_amenities = set(chain.from_iterable(amenity_lists))
work_related = sorted(filter(lambda a: "workspace" in a.lower() or "desk" in a.lower(), all_amenities))
print(len(all_amenities), "distinct amenities;", "work-related:", work_related)

786 distinct amenities; work-related: ['Dedicated workspace', 'Dedicated workspace: desk', 'Dedicated workspace: desk and office chair', 'Dedicated workspace: monitor, desk, and office chair', 'Dedicated workspace: monitor, desk, table, and office chair', 'Dedicated workspace: office chair and desk', 'Dedicated workspace: office chair, desk, and table', 'Dedicated workspace: table', 'Dedicated workspace: table and desk', 'Dedicated workspace: table and office chair', 'Dedicated workspace: table, desk, and office chair', 'Dedicated workspace: table, office chair, and desk']


## 3. Control flow: a small rule with `if / elif / else`
A descriptive label only (not a model feature): how long a stay the host targets.

In [5]:
def stay_type(min_nights: int) -> str:
    if min_nights >= 30:
        return "monthly (30+ nights)"
    elif min_nights >= 7:
        return "weekly (7-29 nights)"
    else:
        return "short (1-6 nights)"

df["minimum_nights"].map(stay_type).value_counts().to_frame("n_listings")

,n_listings
minimum_nights,
monthly (30+ nights),24727
short (1-6 nights),10685
weekly (7-29 nights),1600


## 4. A class: `DataQualityChecker`
It only **reports** problems; it never changes the data (cleaning decisions belong to S7).

In [6]:
checker = DataQualityChecker(df)
quality_report = checker.run(NYC_RULES)
quality_report.to_csv(TABLE_DIR / "03_quality_report.csv", index=False)
print(checker)
quality_report

DataQualityChecker(rows=37012, checks_run=10)


,check,column,n_violations,pct_violations
0,unique listing_id,listing_id,0,0.000
1,price >= 1 USD,price,28,0.076
2,accommodates >= 1 guest,accommodates,13,0.035
3,bedrooms between 0 and 20,bedrooms,1,0.003
4,minimum_nights between 1 and 365,minimum_nights,16,0.043
5,maximum_nights between 1 and 1125 (Airbnb defa...,maximum_nights,18,0.049
6,latitude inside the NYC bounding box,latitude,0,0.000
7,longitude inside the NYC bounding box,longitude,0,0.000
8,host_response_rate between 0 and 1,host_response_rate,0,0.000
9,review_scores_rating between 20 and 100,review_scores_rating,0,0.000


## 5. Error handling
Each call below fails on purpose; `try / except` shows the clear message instead of a crash.

In [7]:
attempts = [
    ("unknown city",        lambda: load_city_listings("Atlantis")),
    ("unknown column",      lambda: checker.check_range(RangeRule("no_such_column", 0, 1))),
    ("not a DataFrame",     lambda: DataQualityChecker([1, 2, 3])),
    ("strict mode",         lambda: checker.assert_clean(NYC_RULES)),
]
for label, call in attempts:
    try:
        call()
        print(f"{label:16s} -> no error")
    except (ValueError, TypeError, DataQualityError) as exc:
        print(f"{label:16s} -> {type(exc).__name__}: {str(exc)[:110]}")

unknown city     -> ValueError: No listings found for city='Atlantis'. Known cities: Bangkok, Cape Town, Hong Kong, Istanbul, Mexico City, New
unknown column   -> DataQualityError: Column 'no_such_column' not found for rule RangeRule(column='no_such_column', min_value=0, max_value=1, descri
not a DataFrame  -> TypeError: Expected a pandas DataFrame, got list
strict mode      -> DataQualityError: 5 check(s) failed: ['price >= 1 USD', 'accommodates >= 1 guest', 'bedrooms between 0 and 20', 'minimum_nights 


## 6. Tests
`tests/test_src.py` checks the functions and the class on tiny synthetic tables, so the tests also run without the raw data.

In [8]:
import importlib
import tests.test_src as test_src
importlib.reload(test_src)

test_results = test_src.run_all()
print(test_results["result"].value_counts().to_string())
test_results

result
PASS    11


,test,result,error
0,test_find_tf_columns,PASS,
1,test_to_bool_keeps_missing,PASS,
2,test_parse_amenities_valid,PASS,
3,test_parse_amenities_bad_input,PASS,
4,test_missing_report,PASS,
5,test_checker_finds_duplicate_ids,PASS,
6,test_checker_range_ignores_missing,PASS,
7,test_checker_unknown_column_raises,PASS,
8,test_checker_rejects_non_dataframe,PASS,
9,test_assert_clean_strict_mode,PASS,


## 7. Clean-clone check (Git/GitHub)
A project is reproducible only if a **fresh copy from GitHub** runs. This cell:
1. downloads the `main` branch of the public repository as a zip into a temporary folder;
2. links our local `data/raw` into it (simulating "download the Kaggle data" from the README);
3. runs the tests and executes notebooks 01 and 02 inside that fresh copy.

In [9]:
import io, os, shutil, subprocess, tempfile, time, urllib.request, zipfile
import nbformat
from nbclient import NotebookClient

REPO_ZIP = "https://github.com/alisahann/nyc-airbnb-ds/archive/refs/heads/main.zip"
tmp = Path(tempfile.mkdtemp(prefix="clean_clone_"))
with urllib.request.urlopen(REPO_ZIP, timeout=60) as resp:
    zipfile.ZipFile(io.BytesIO(resp.read())).extractall(tmp)
clone = next(tmp.iterdir())
print("Fresh copy:", clone.name, "| files:", sorted(p.name for p in clone.iterdir()))

(clone / "data").mkdir(exist_ok=True)
os.symlink(PROJECT_ROOT / "data" / "raw", clone / "data" / "raw")   # stands in for the Kaggle download

checks = []
t = subprocess.run([sys.executable, "-m", "tests.test_src"], cwd=clone, capture_output=True, text=True)
checks.append({"step": "tests/test_src.py", "ok": t.returncode == 0, "detail": t.stdout.strip().splitlines()[-1] if t.stdout else t.stderr[-200:]})

for nb_name in ["01_first_look.ipynb", "02_problem_definition.ipynb"]:
    nb_path = clone / "notebooks" / nb_name
    nb = nbformat.read(nb_path, as_version=4)
    start = time.time()
    try:
        NotebookClient(nb, timeout=900, kernel_name="python3",
                       resources={"metadata": {"path": str(nb_path.parent)}}).execute()
        checks.append({"step": nb_name, "ok": True, "detail": f"ran in {time.time() - start:.0f} s"})
    except Exception as exc:
        checks.append({"step": nb_name, "ok": False, "detail": f"{type(exc).__name__}: {str(exc)[:200]}"})

shutil.rmtree(tmp, ignore_errors=True)
pd.DataFrame(checks)

Fresh copy: nyc-airbnb-ds-main | files: ['.gitignore', 'README.md', 'configs', 'environment.yml', 'notebooks', 'reports', 'src', 'tests']


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


,step,ok,detail
0,tests/test_src.py,True,test_load_missing_file_raises PASS
1,01_first_look.ipynb,True,ran in 6 s
2,02_problem_definition.ipynb,True,ran in 4 s


## 8. Evaluation of S3

**What we did**
- Moved repeated code into modules: `src/data_utils.py` (loading, t/f conversion, amenity parsing, missing report) and `src/quality.py` (`DataQualityChecker` class, `RangeRule` dataclass, custom `DataQualityError`).
- Covered the course's Python topics on real project data: data structures, functions, `if/elif/else`, a class, `map` / `filter` / `lambda`, modules and error handling.
- Wrote 11 tests on small synthetic tables (`tests/test_src.py`): **11 / 11 PASS**.
- **Clean-clone check passed:** a fresh copy of the GitHub repository ran the tests and executed notebooks 01 and 02 without errors.

**Findings from the toolkit (for later stages)**
- 4 columns are t/f flags (`host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `instant_bookable`); `host_has_profile_pic` is 99.7% True, so it carries almost no information.
- 786 distinct amenities, median 19 per listing; Wifi (97%), "Long term stays allowed" (94%) and Heating (94%) are nearly universal → only rarer amenities can separate listings.
- 24,727 of 37,012 listings (66.8%) require 30+ nights: the snapshot is dominated by monthly rentals (COVID period and NYC short-term rental rules). This matters for how we interpret "nightly price".
- Plausibility report (`reports/tables/03_quality_report.csv`): 28 prices below 1 USD, 13 listings for 0 guests, 1 listing with more than 20 bedrooms, 16 with `minimum_nights` > 365, 18 with `maximum_nights` > 1,125. Coordinates, response rates and rating ranges are all valid. Nothing was changed; treatment is decided in S7.

**Decisions (Tier 1, no review needed):** module layout; tests written without a pytest dependency so they run with plain Python; the quality checker only reports and never edits data.

**Next step:** S4 – data loading and manipulation with Pandas (selection, conditional operations, groupby / aggregate / filter / transform / apply, pivot tables, a merge with the review data).
